# CSE 151B — Full Submission Pipeline

End-to-end pipeline to produce `submission.csv` for `data/private.jsonl`, combining:

1. **Prompt engineering** — chain-of-thought + few-shot exemplars + self-consistency (majority vote) + progressive-hint prompting.
2. **Supervised fine-tuning (LoRA / QLoRA)** — distil the same base model on the public set (correct rationales kept via rejection sampling).
3. **Preference optimisation (DPO)** — pair self-generated correct vs. incorrect traces as `(chosen, rejected)` for further alignment.
4. **Inference-time scaling** — batched vLLM generation of N samples per question, majority-vote answer extraction, progressive-hint retry for low-confidence items.

Every heavy stage is gated by a boolean flag at the top of its section, so you can run prompt-only first to get a quick baseline submission and then turn training stages on incrementally.

Reuses the project `.venv` from `starter_code_cse151b_comp.ipynb`. Select the **Python (cse151b)** kernel before running.

## 0. Activate venv and install extra training deps

Only `peft`, `trl`, `datasets`, `accelerate` need adding on top of the starter env. Run once, then comment out.

In [1]:
# One-time install (uncomment on first run, then re-comment to skip).
# !.venv/bin/python -m pip install -q --upgrade peft trl datasets accelerate
!source ./.venv/bin/activate && echo "venv ready"

venv ready


## 1. Configuration

All knobs in one place. Flip the `RUN_*` switches to enable training stages.

In [ ]:
import os, json, re, sys, gc, csv, math, random, time
from pathlib import Path
from typing import Optional, List, Tuple
from collections import Counter, defaultdict

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_MODEL_ID       = "Qwen/Qwen3-4B-Thinking-2507"
PUBLIC_PATH         = "data/public.jsonl"
PRIVATE_PATH        = "data/private.jsonl"
RESULTS_DIR         = Path("results");        RESULTS_DIR.mkdir(exist_ok=True)
TRAIN_DIR           = Path("training");       TRAIN_DIR.mkdir(exist_ok=True)
SFT_DATA_PATH       = TRAIN_DIR / "sft_traces.jsonl"
DPO_DATA_PATH       = TRAIN_DIR / "dpo_pairs.jsonl"
SFT_LORA_DIR        = TRAIN_DIR / "sft_lora"
DPO_LORA_DIR        = TRAIN_DIR / "dpo_lora"
MERGED_MODEL_DIR    = TRAIN_DIR / "merged_model"
SUBMISSION_CSV      = Path("submission.csv")
PRIVATE_GEN_PATH    = RESULTS_DIR / "private_generations.jsonl"

# ── Stage switches ────────────────────────────────────────────────────────────
RUN_BUILD_SFT_DATA  = False   # rejection-sample teacher rationales on public set
RUN_SFT_LORA        = False   # QLoRA SFT on the rationales above
RUN_BUILD_DPO_DATA  = False   # build (chosen, rejected) preference pairs
RUN_DPO             = False   # DPO on top of SFT-LoRA
RUN_MERGE_LORA      = False   # merge adapter into base for vLLM loading
RUN_PRIVATE_INFER   = True    # always-on: generate the submission

# ── Hardware / runtime ────────────────────────────────────────────────────────
GPU_ID              = "0"
os.environ["CUDA_VISIBLE_DEVICES"]      = GPU_ID
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
os.environ["TOKENIZERS_PARALLELISM"]    = "false"

# ── Inference hyperparameters ─────────────────────────────────────────────────
MAX_MODEL_LEN       = 16384
MAX_NEW_TOKENS      = 12288
N_SAMPLES_PER_Q     = 5       # self-consistency samples per question
PROGRESSIVE_HINT_K  = 1       # extra rounds of progressive-hint prompting for low-confidence Qs
CONFIDENCE_TAU      = 0.5     # if top vote fraction < tau, trigger progressive-hint round
FEW_SHOT_K_MCQ      = 2
FEW_SHOT_K_FREE     = 2
TEMPERATURE         = 0.7
TOP_P               = 0.95
TOP_K               = 20

# ── Training hyperparameters ──────────────────────────────────────────────────
SFT_REJECTION_N     = 4       # generations per public question when building SFT data
SFT_EPOCHS          = 1
SFT_LR              = 2e-4
SFT_BSZ             = 1
SFT_GRAD_ACCUM      = 16
DPO_EPOCHS          = 1
DPO_LR              = 5e-6
DPO_BETA            = 0.1

random.seed(0)
print("Config loaded.")

## 2. Load datasets and the project judger

In [ ]:
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

def load_jsonl(p):
    return [json.loads(l) for l in open(p)]

public_data  = load_jsonl(PUBLIC_PATH)
private_data = load_jsonl(PRIVATE_PATH)

def n_blanks(item):
    return max(1, item["question"].count("[ANS]")) if not item.get("options") else 1

print(f"public:  {len(public_data)}  (mcq={sum(bool(d.get('options')) for d in public_data)})")
print(f"private: {len(private_data)} (mcq={sum(bool(d.get('options')) for d in private_data)})")

## 3. Prompt engineering

* **Chain-of-thought** — system prompt instructs step-by-step reasoning before `\boxed{}`.
* **Few-shot exemplars** — short worked examples from the public set, one MCQ batch and one free-form batch. Selected to be short so they don't blow the context.
* **Progressive-hint** — when self-consistency confidence is low, we re-prompt the model with its own top candidate answer as a hint ("Hint: previous answers were {X}. Verify and either confirm or correct.").
* **Self-consistency** — sample N traces per question, extract answers, majority-vote.

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Reason carefully and step-by-step. "
    "At the very end, output ONLY the final answer wrapped in a single \\boxed{}. "
    "If the problem has multiple [ANS] placeholders, output ALL final answers in order, "
    "comma-separated, inside ONE \\boxed{}, e.g. \\boxed{3, 7, -1}. "
    "Do not add units inside \\boxed{} unless the question explicitly asks. "
    "Simplify radicals and fractions where possible."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. Read the problem and choices, reason step-by-step, "
    "then output ONLY the letter of the single best option inside \\boxed{}, e.g. \\boxed{C}. "
    "Output exactly one letter; do not output the option text."
)

# Curate compact few-shot exemplars from the public set (short questions only,
# trustworthy answers). These are model-agnostic and live in the user turn.
def _short(item, lim_q=350, lim_a=80):
    if len(item["question"]) > lim_q: return False
    if item.get("options"):
        return len(str(item.get("answer", ""))) <= 3 and all(len(o) <= lim_a for o in item["options"])
    a = item.get("answer", [])
    if not isinstance(a, list): a = [a]
    return len(a) <= 2 and all(len(str(x)) <= 30 for x in a)

_pub_mcq_short  = [d for d in public_data if d.get("options") and _short(d)]
_pub_free_short = [d for d in public_data if not d.get("options") and _short(d)]
random.Random(7).shuffle(_pub_mcq_short)
random.Random(7).shuffle(_pub_free_short)
FEW_SHOT_MCQ  = _pub_mcq_short[:FEW_SHOT_K_MCQ]
FEW_SHOT_FREE = _pub_free_short[:FEW_SHOT_K_FREE]

def _format_choices(options):
    labels = [chr(65 + i) for i in range(len(options))]
    return "\n".join(f"{l}. {o.strip()}" for l, o in zip(labels, options))

def _exemplar_block(items, is_mcq):
    if not items: return ""
    lines = ["Worked examples (study the format, then solve the new question):\n"]
    for ex in items:
        q = ex["question"].strip()
        if is_mcq:
            ans = str(ex["answer"]).strip().upper()
            lines.append(f"Example question:\n{q}\n\nOptions:\n{_format_choices(ex['options'])}\n\nFinal answer: \\boxed{{{ans}}}\n")
        else:
            ans = ex["answer"] if isinstance(ex["answer"], list) else [ex["answer"]]
            lines.append(f"Example question:\n{q}\n\nFinal answer: \\boxed{{{', '.join(str(a) for a in ans)}}}\n")
    lines.append("Now solve the NEW question below. Show your reasoning then finish with \\boxed{...}.\n")
    return "\n".join(lines)

def build_messages(item, hints: Optional[List[str]] = None):
    """Return list[dict] chat messages with CoT + few-shot + optional progressive hints."""
    is_mcq = bool(item.get("options"))
    system = SYSTEM_PROMPT_MCQ if is_mcq else SYSTEM_PROMPT_MATH
    ex_block = _exemplar_block(FEW_SHOT_MCQ if is_mcq else FEW_SHOT_FREE, is_mcq)
    user_parts = [ex_block] if ex_block else []
    user_parts.append("New question:\n" + item["question"].strip())
    if is_mcq:
        user_parts.append("\nOptions:\n" + _format_choices(item["options"]))
    if hints:
        hint_str = ", ".join(str(h) for h in hints)
        user_parts.append(
            f"\nHint: previous attempts produced the candidate answer(s) {{{hint_str}}}. "
            "Reconsider the problem from scratch; if you agree, confirm; otherwise correct it. "
            "Output the final answer in \\boxed{} as instructed."
        )
    return [
        {"role": "system", "content": system},
        {"role": "user",   "content": "\n".join(user_parts)},
    ]

print(f"Few-shot MCQ exemplars : {len(FEW_SHOT_MCQ)}")
print(f"Few-shot free exemplars: {len(FEW_SHOT_FREE)}")

## 4. Answer extraction + self-consistency voting

In [ ]:
from utils import last_boxed_only_string, remove_boxed

_LETTER_RE = re.compile(r"\\boxed\{\s*([A-Za-z])\s*\}")

def extract_letter(text: str) -> str:
    m = _LETTER_RE.search(text or "")
    if m: return m.group(1).upper()
    # Fallback: last standalone capital letter on the last non-empty line
    for line in reversed((text or "").splitlines()):
        line = line.strip()
        if not line: continue
        ms = re.findall(r"\b([A-Z])\b", line)
        if ms: return ms[-1]
    return ""

def extract_boxed_inner(text: str) -> str:
    if not text: return ""
    boxed = last_boxed_only_string(text)
    inner = remove_boxed(boxed) if boxed else None
    return (inner or "").strip()

def normalize_free_answer(text: str) -> str:
    """Return a canonical string for voting on free-form responses."""
    s = extract_boxed_inner(text)
    if not s: return ""
    s = s.replace(" ", "").replace("$", "")
    s = s.replace("\\,", "").replace("\\;", "").replace("\\!", "")
    return s

def vote(responses: List[str], is_mcq: bool) -> Tuple[str, float, str]:
    """Return (canonical_answer, top_fraction, best_response_text)."""
    if not responses: return "", 0.0, ""
    if is_mcq:
        keys = [extract_letter(r) for r in responses]
    else:
        keys = [normalize_free_answer(r) for r in responses]
    counts = Counter(k for k in keys if k)
    if not counts:
        return "", 0.0, responses[0]
    top_key, top_n = counts.most_common(1)[0]
    frac = top_n / len(responses)
    # Pick the longest response that voted for the winning key as the canonical trace.
    candidates = [r for r, k in zip(responses, keys) if k == top_key]
    best = max(candidates, key=len)
    return top_key, frac, best

## 5. Build SFT data via rejection sampling (optional)

We solve each public question with the base model `SFT_REJECTION_N` times, keep traces whose extracted answer the `Judger` accepts as correct, and write `(prompt, completion)` pairs to `training/sft_traces.jsonl`.

In [ ]:
def chat_template_prompt(tokenizer, messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def judge_response(item, response: str) -> bool:
    is_mcq = bool(item.get("options"))
    if is_mcq:
        return extract_letter(response) == str(item["answer"]).strip().upper()
    gold = item["answer"] if isinstance(item["answer"], list) else [item["answer"]]
    try:
        return bool(judger.auto_judge(pred=response, gold=gold, options=[[]] * len(gold)))
    except Exception:
        return False

if RUN_BUILD_SFT_DATA:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    llm = LLM(
        model=BASE_MODEL_ID,
        quantization="bitsandbytes", load_format="bitsandbytes",
        gpu_memory_utilization=0.50, max_model_len=MAX_MODEL_LEN,
        trust_remote_code=True, max_num_seqs=128, enable_prefix_caching=True,
    )
    sp = SamplingParams(
        n=SFT_REJECTION_N, max_tokens=MAX_NEW_TOKENS,
        temperature=0.8, top_p=0.95, top_k=20,
    )

    prompts = [chat_template_prompt(tok, build_messages(it)) for it in public_data]
    print(f"Generating {SFT_REJECTION_N}x{len(prompts)} traces …")
    outs = llm.generate(prompts, sampling_params=sp)

    n_kept = 0
    with open(SFT_DATA_PATH, "w") as f:
        for item, prompt, out in zip(public_data, prompts, outs):
            for cand in out.outputs:
                trace = cand.text.strip()
                if judge_response(item, trace):
                    f.write(json.dumps({"prompt": prompt, "completion": trace, "id": item["id"]}) + "\n")
                    n_kept += 1
                    break   # one correct trace per question is enough for SFT
    print(f"Kept {n_kept}/{len(public_data)} correct traces -> {SFT_DATA_PATH}")

    # Free vLLM before training.
    del llm; gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception: pass
else:
    print("Skipping SFT data build (RUN_BUILD_SFT_DATA=False).")

## 6. QLoRA supervised fine-tuning (optional)

4-bit base, LoRA adapters on attention + MLP projections, trained with `trl.SFTTrainer` on the rejection-sampled traces.

In [ ]:
if RUN_SFT_LORA:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, prepare_model_for_kbit_training
    from trl import SFTTrainer, SFTConfig
    from datasets import load_dataset

    assert SFT_DATA_PATH.exists(), "Run section 5 first."
    ds = load_dataset("json", data_files=str(SFT_DATA_PATH), split="train")

    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4",
    )
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, quantization_config=bnb, device_map="auto", trust_remote_code=True,
    )
    base = prepare_model_for_kbit_training(base)

    lora_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
    )

    def fmt(ex):
        # Concatenate prompt + completion; SFTTrainer will mask the prompt if we want,
        # but here we let it learn the full sequence (standard distillation).
        return {"text": ex["prompt"] + ex["completion"] + tok.eos_token}
    ds = ds.map(fmt, remove_columns=ds.column_names)

    cfg = SFTConfig(
        output_dir=str(SFT_LORA_DIR),
        per_device_train_batch_size=SFT_BSZ,
        gradient_accumulation_steps=SFT_GRAD_ACCUM,
        num_train_epochs=SFT_EPOCHS,
        learning_rate=SFT_LR,
        bf16=True, logging_steps=10, save_strategy="epoch",
        max_length=MAX_MODEL_LEN, packing=False, report_to="none",
        gradient_checkpointing=True, optim="paged_adamw_8bit",
        dataset_text_field="text",
    )
    trainer = SFTTrainer(model=base, args=cfg, train_dataset=ds,
                          peft_config=lora_cfg, processing_class=tok)
    trainer.train()
    trainer.save_model(str(SFT_LORA_DIR))
    print("SFT-LoRA saved ->", SFT_LORA_DIR)

    del trainer, base; gc.collect(); torch.cuda.empty_cache()
else:
    print("Skipping SFT-LoRA (RUN_SFT_LORA=False).")

## 7. Build DPO preference pairs (optional)

Use the same SFT generations (or generate fresh ones) and pair `chosen = judged-correct trace`, `rejected = judged-incorrect trace` per question.

In [ ]:
if RUN_BUILD_DPO_DATA:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    llm = LLM(
        model=BASE_MODEL_ID,
        quantization="bitsandbytes", load_format="bitsandbytes",
        gpu_memory_utilization=0.50, max_model_len=MAX_MODEL_LEN,
        trust_remote_code=True, max_num_seqs=128, enable_prefix_caching=True,
    )
    sp = SamplingParams(n=6, max_tokens=MAX_NEW_TOKENS,
                         temperature=0.9, top_p=0.95, top_k=40)
    prompts = [chat_template_prompt(tok, build_messages(it)) for it in public_data]
    outs = llm.generate(prompts, sampling_params=sp)

    n_pairs = 0
    with open(DPO_DATA_PATH, "w") as f:
        for item, prompt, out in zip(public_data, prompts, outs):
            good, bad = None, None
            for cand in out.outputs:
                t = cand.text.strip()
                if judge_response(item, t):
                    good = good or t
                else:
                    bad  = bad or t
                if good and bad: break
            if good and bad:
                f.write(json.dumps({"prompt": prompt, "chosen": good, "rejected": bad, "id": item["id"]}) + "\n")
                n_pairs += 1
    print(f"Built {n_pairs} DPO pairs -> {DPO_DATA_PATH}")
    del llm; gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception: pass
else:
    print("Skipping DPO data build (RUN_BUILD_DPO_DATA=False).")

## 8. DPO alignment on top of SFT adapter (optional)

In [ ]:
if RUN_DPO:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, PeftModel
    from trl import DPOTrainer, DPOConfig
    from datasets import load_dataset

    assert DPO_DATA_PATH.exists(), "Run section 7 first."
    ds = load_dataset("json", data_files=str(DPO_DATA_PATH), split="train")

    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4",
    )
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, quantization_config=bnb, device_map="auto", trust_remote_code=True,
    )
    if SFT_LORA_DIR.exists():
        policy = PeftModel.from_pretrained(base, str(SFT_LORA_DIR), is_trainable=True)
    else:
        policy = base

    lora_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
    )

    cfg = DPOConfig(
        output_dir=str(DPO_LORA_DIR),
        per_device_train_batch_size=1, gradient_accumulation_steps=16,
        num_train_epochs=DPO_EPOCHS, learning_rate=DPO_LR, beta=DPO_BETA,
        bf16=True, logging_steps=10, save_strategy="epoch",
        max_length=MAX_MODEL_LEN, max_prompt_length=MAX_MODEL_LEN // 2,
        report_to="none", gradient_checkpointing=True, optim="paged_adamw_8bit",
    )
    trainer = DPOTrainer(model=policy, ref_model=None, args=cfg, train_dataset=ds,
                          processing_class=tok, peft_config=lora_cfg if not SFT_LORA_DIR.exists() else None)
    trainer.train()
    trainer.save_model(str(DPO_LORA_DIR))
    print("DPO-LoRA saved ->", DPO_LORA_DIR)
    del trainer, policy, base; gc.collect(); torch.cuda.empty_cache()
else:
    print("Skipping DPO (RUN_DPO=False).")

## 9. Merge adapter into a vLLM-loadable checkpoint (optional)

vLLM cannot consume QLoRA adapters directly while the base is in 4-bit; we merge the latest adapter into BF16 weights for inference.

In [ ]:
if RUN_MERGE_LORA:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    adapter_dir = DPO_LORA_DIR if DPO_LORA_DIR.exists() else SFT_LORA_DIR
    assert adapter_dir.exists(), "No adapter to merge."
    print("Merging adapter from", adapter_dir)

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True,
    )
    merged = PeftModel.from_pretrained(base, str(adapter_dir)).merge_and_unload()
    merged.save_pretrained(MERGED_MODEL_DIR, safe_serialization=True)
    AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True).save_pretrained(MERGED_MODEL_DIR)
    print("Merged model ->", MERGED_MODEL_DIR)
    del base, merged; gc.collect(); torch.cuda.empty_cache()
else:
    print("Skipping merge (RUN_MERGE_LORA=False).")

## 10. Final inference on `private.jsonl` with self-consistency + progressive hints

Loads the merged fine-tuned checkpoint if it exists, otherwise falls back to the base model. Generates `N_SAMPLES_PER_Q` traces per question in one batched vLLM call, votes, and for items where the top vote share is below `CONFIDENCE_TAU` we run an extra progressive-hint round using the current top answer as a hint.

In [ ]:
if RUN_PRIVATE_INFER:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    from tqdm import tqdm

    model_path = str(MERGED_MODEL_DIR) if MERGED_MODEL_DIR.exists() else BASE_MODEL_ID
    use_bnb    = (model_path == BASE_MODEL_ID)
    print("Inference model:", model_path, "| quant:", "int8-bnb" if use_bnb else "bf16")

    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    llm_kwargs = dict(
        model=model_path, trust_remote_code=True,
        gpu_memory_utilization=0.55, max_model_len=MAX_MODEL_LEN,
        max_num_seqs=128, max_num_batched_tokens=32768,
        enable_prefix_caching=True,
    )
    if use_bnb:
        llm_kwargs.update(quantization="bitsandbytes", load_format="bitsandbytes")

    llm = LLM(**llm_kwargs)

    sp_main = SamplingParams(
        n=N_SAMPLES_PER_Q, max_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
    )
    sp_hint = SamplingParams(
        n=max(2, N_SAMPLES_PER_Q // 2), max_tokens=MAX_NEW_TOKENS,
        temperature=max(0.4, TEMPERATURE - 0.2), top_p=TOP_P, top_k=TOP_K,
    )

    # ── Round 1: self-consistency ────────────────────────────────────────────
    prompts_r1 = [chat_template_prompt(tok, build_messages(it)) for it in private_data]
    print(f"Round 1: generating {N_SAMPLES_PER_Q} samples for {len(prompts_r1)} questions …")
    outs_r1 = llm.generate(prompts_r1, sampling_params=sp_main)

    per_q_samples = []
    voted = []
    for item, out in zip(private_data, outs_r1):
        traces = [c.text.strip() for c in out.outputs]
        is_mcq = bool(item.get("options"))
        key, frac, best = vote(traces, is_mcq)
        per_q_samples.append(traces)
        voted.append({"id": item["id"], "is_mcq": is_mcq, "key": key,
                       "frac": frac, "best": best})

    # ── Progressive-hint round for low-confidence items ──────────────────────
    for k in range(PROGRESSIVE_HINT_K):
        low_idx = [i for i, v in enumerate(voted)
                   if v["frac"] < CONFIDENCE_TAU and v["key"]]
        if not low_idx:
            print(f"Hint round {k+1}: nothing below tau={CONFIDENCE_TAU}, done.")
            break
        print(f"Hint round {k+1}: re-prompting {len(low_idx)}/{len(voted)} low-confidence items")
        hint_prompts = [
            chat_template_prompt(tok, build_messages(private_data[i], hints=[voted[i]["key"]]))
            for i in low_idx
        ]
        outs_h = llm.generate(hint_prompts, sampling_params=sp_hint)
        for slot, out in zip(low_idx, outs_h):
            item = private_data[slot]
            traces = per_q_samples[slot] + [c.text.strip() for c in out.outputs]
            per_q_samples[slot] = traces
            key, frac, best = vote(traces, voted[slot]["is_mcq"])
            voted[slot] = {"id": item["id"], "is_mcq": voted[slot]["is_mcq"],
                            "key": key, "frac": frac, "best": best}

    # ── Persist intermediate per-question data for debugging ─────────────────
    with open(PRIVATE_GEN_PATH, "w") as f:
        for item, v, traces in zip(private_data, voted, per_q_samples):
            f.write(json.dumps({"id": item["id"], "is_mcq": v["is_mcq"],
                                  "voted_key": v["key"], "vote_frac": v["frac"],
                                  "n_samples": len(traces),
                                  "best_response": v["best"]}) + "\n")
    print("Saved per-question generations ->", PRIVATE_GEN_PATH)

    del llm; gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception: pass
else:
    print("Skipping private inference (RUN_PRIVATE_INFER=False).")

## 11. Write the submission CSV

The grader extracts the final answer from the `\boxed{}` in the response trace. We:

1. Start from the best voted trace.
2. Defensively ensure it ends with a `\boxed{...}` containing the voted answer — if the model's own `\boxed{}` doesn't match the voted key (which can happen for free-form), append a final `\boxed{voted_key}` so the extracted answer matches the majority vote.
3. Write `id,response` with proper CSV quoting.

In [ ]:
def finalize_response(v):
    text = v["best"] or ""
    key  = v["key"]
    if not key:
        return text if text else "\\boxed{}"
    if v["is_mcq"]:
        if extract_letter(text) == key:
            return text
        return (text.rstrip() + f"\n\nFinal answer: \\boxed{{{key}}}").strip()
    # free-form: compare normalized boxed content
    if normalize_free_answer(text) == key and key:
        return text
    return (text.rstrip() + f"\n\nFinal answer: \\boxed{{{key}}}").strip()

# voted may not exist if section 10 was skipped — guard for that.
if "voted" not in globals():
    raise RuntimeError("Run section 10 first to produce `voted`.")

assert len(voted) == len(private_data), "voted length mismatch"

with open(SUBMISSION_CSV, "w", newline="") as f:
    w = csv.writer(f, quoting=csv.QUOTE_ALL)
    w.writerow(["id", "response"])
    for item, v in zip(private_data, voted):
        w.writerow([item["id"], finalize_response(v)])

print(f"Wrote {SUBMISSION_CSV} with {len(private_data)} rows.")

## 12. (Optional) Sanity-check accuracy on the public set

Run the same prompt+self-consistency pipeline on `public.jsonl` to estimate your private-set accuracy before submitting.

In [ ]:
RUN_PUBLIC_EVAL = False

if RUN_PUBLIC_EVAL:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    model_path = str(MERGED_MODEL_DIR) if MERGED_MODEL_DIR.exists() else BASE_MODEL_ID
    use_bnb    = (model_path == BASE_MODEL_ID)
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    tok.pad_token = tok.eos_token
    kw = dict(model=model_path, trust_remote_code=True,
               gpu_memory_utilization=0.55, max_model_len=MAX_MODEL_LEN,
               max_num_seqs=128, enable_prefix_caching=True)
    if use_bnb: kw.update(quantization="bitsandbytes", load_format="bitsandbytes")
    llm = LLM(**kw)
    sp = SamplingParams(n=N_SAMPLES_PER_Q, max_tokens=MAX_NEW_TOKENS,
                         temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K)
    prompts = [chat_template_prompt(tok, build_messages(it)) for it in public_data]
    outs = llm.generate(prompts, sampling_params=sp)

    n_ok = 0
    for item, out in zip(public_data, outs):
        traces = [c.text.strip() for c in out.outputs]
        is_mcq = bool(item.get("options"))
        key, frac, best = vote(traces, is_mcq)
        synth = finalize_response({"best": best, "key": key, "is_mcq": is_mcq})
        n_ok += int(judge_response(item, synth))
    print(f"Public accuracy: {n_ok}/{len(public_data)} = {n_ok/len(public_data)*100:.2f}%")
    del llm; gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception: pass
else:
    print("Set RUN_PUBLIC_EVAL=True to estimate accuracy on the public set.")